# S08 · Shrink the weights — Ridge, Lasso and ElasticNet

We start from the plain model you built in Session 7, then add a gentle rule that
stops it over-trusting any one feature. We watch two styles of that rule: Ridge
quietly shrinks every weight, Lasso switches the useless ones off completely.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the idea of a model "over-trusting" the data? Open the primer
  `primers/overfitting_and_regularization.md` for a ten-minute, picture-first
  version.
- Already confident with code or with shrinkage methods? Skip to the cell marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scikit-learn.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                    # fast maths on lists of numbers
import matplotlib.pyplot as plt                        # drawing charts
from sklearn.linear_model import LinearRegression      # the plain model from Session 7
from sklearn.linear_model import Ridge                 # shrink every weight (L2 penalty)
from sklearn.linear_model import Lasso                 # switch useless weights off (L1 penalty)
from sklearn.linear_model import ElasticNet            # a blend of the two

## Step 1 — make some flats where we know the truth

The easiest way to see over-trusting happen is to build data where we already know
which attributes matter. We invent 80 flats, each described by 7 everyday
attributes: `size_sqft`, `age_years`, `locality_score`, `floor`,
`distance_to_metro_km`, `num_bathrooms` and `has_parking`. We decide the real
rule: only the first three (size, age and locality) actually affect the price. The
other four are pure noise in this made-up market, like the seller's favourite
colour. A good model should learn to ignore them.

In [ ]:
# Set a seed so everyone gets the same "random" flats and the same result.
np.random.seed(0)

# 80 flats (rows), each described by 7 measurements (columns).
number_of_rows = 80
number_of_features = 7

# Give each of the 7 columns a real, everyday flat attribute, so the weights we
# learn later read as "price per unit of a real thing", not an abstract vector.
feature_names = [
    "size_sqft",             # floor area in square feet
    "age_years",             # how old the flat is
    "locality_score",        # how desirable the neighbourhood is
    "floor",                 # which storey the flat is on
    "distance_to_metro_km",  # walking distance to the nearest metro
    "num_bathrooms",         # number of bathrooms
    "has_parking",           # 1 if a parking spot is included, else 0
]

# Each measurement is just a random number here.
features = np.random.normal(0, 1, size=(number_of_rows, number_of_features))

# The TRUE rule this made-up market follows: only the first three attributes
# (size_sqft, age_years, locality_score) actually move the price. Bigger flats
# and nicer areas cost more (+), older flats cost less (-). The other four
# attributes have a true weight of 0 - in this toy market they are pure noise.
true_weights = np.array([5.0, -3.0, 2.0, 0.0, 0.0, 0.0, 0.0])

# Build the price from the true rule, plus a little random wobble.
noise = np.random.normal(0, 1.0, size=number_of_rows)
target = features @ true_weights + noise

print("features shape:", features.shape)
for column_number in range(number_of_features):
    print("  ", feature_names[column_number].ljust(20), "true weight:", true_weights[column_number])

print("\nfloor, distance_to_metro_km, num_bathrooms and has_parking should not")
print("matter at all. Let us see which model notices.")

## Step 2 — fit the plain model and watch it over-trust

First the model from Session 7: ordinary linear regression, with no gentle rule at
all. Look closely at the weights it learns for the four useless columns. They will
not be exactly zero. The model has quietly decided that noise means something,
which is exactly the over-trusting we want to stop.

In [ ]:
# Create the plain model and fit it on all the flats.
plain_model = LinearRegression()
plain_model.fit(features, target)

# The learned weight for each column.
plain_coefficients = plain_model.coef_

print("Plain linear regression weights:")
for column_number in range(number_of_features):
    print("  ", feature_names[column_number].ljust(20), ":", round(plain_coefficients[column_number], 3))

print("\nThe true weight of floor, distance_to_metro_km, num_bathrooms and")
print("has_parking is 0, but the plain model gave them nonzero weights.")
print("It over-trusted the noise.")

## Step 3 — Ridge: shrink every weight

Ridge adds a small fee for making any weight large. The fee is the sum of the
squared weights, scaled by the knob `alpha`. Turning that knob up makes the model
pay more for loud weights, so it quietens all of them. Ridge pulls every weight
toward zero, but never all the way to exactly zero. It just turns the volume
down.

In [ ]:
# alpha is the knob: bigger alpha means a stronger fee, so more shrinkage.
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(features, target)

ridge_coefficients = ridge_model.coef_

print("Ridge weights (alpha = 10):")
for column_number in range(number_of_features):
    print("  ", feature_names[column_number].ljust(20), ":", round(ridge_coefficients[column_number], 3))

print("\nEvery weight is a bit smaller than the plain model, but none is exactly 0.")

## Step 4 — Lasso: switch useless weights off

Lasso charges its fee a different way: the sum of the *absolute* sizes of the
weights. That small change has a dramatic effect. Instead of just shrinking, Lasso
pushes some weights to *exactly* zero, which drops those features from the model
entirely. On our data it should switch off the four useless columns by itself. That
is automatic feature selection.

In [ ]:
# Same knob idea. Lasso is the one that can produce exact zeros.
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(features, target)

lasso_coefficients = lasso_model.coef_

print("Lasso weights (alpha = 0.1):")
for column_number in range(number_of_features):
    print("  ", feature_names[column_number].ljust(20), ":", round(lasso_coefficients[column_number], 3))

print("\nLook: the noise attributes (floor, distance_to_metro_km, num_bathrooms,")
print("has_parking) are now exactly 0.0 - Lasso dropped them.")

## Step 5 — ElasticNet: a blend of the two

ElasticNet uses both fees at once. The `l1_ratio` sets the blend: `1.0` is pure
Lasso, `0.0` is pure Ridge, and values in between mix them. It can still switch
features off, while behaving more gracefully when features move together.

In [ ]:
# l1_ratio = 0.5 means half Lasso, half Ridge.
elasticnet_model = ElasticNet(alpha=0.1, l1_ratio=0.5)
elasticnet_model.fit(features, target)

elasticnet_coefficients = elasticnet_model.coef_

print("ElasticNet weights (alpha = 0.1, l1_ratio = 0.5):")
for column_number in range(number_of_features):
    print("  ", feature_names[column_number].ljust(20), ":", round(elasticnet_coefficients[column_number], 3))

## Step 6 — compare all four side by side

A bar chart makes the difference obvious. Watch how Ridge keeps every bar (just
shorter) while Lasso and ElasticNet flatten the useless columns to zero.

In [ ]:
# The positions along the bottom, one per attribute.
column_numbers = np.arange(number_of_features)

# We draw four sets of bars next to each other, so we shift each set sideways.
bar_width = 0.2

plt.figure(figsize=(10, 5))
plt.bar(column_numbers - 1.5 * bar_width, plain_coefficients, width=bar_width, label="Plain")
plt.bar(column_numbers - 0.5 * bar_width, ridge_coefficients, width=bar_width, label="Ridge (shrink)")
plt.bar(column_numbers + 0.5 * bar_width, lasso_coefficients, width=bar_width, label="Lasso (drop)")
plt.bar(column_numbers + 1.5 * bar_width, elasticnet_coefficients, width=bar_width, label="ElasticNet")

plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("flat attribute")
plt.ylabel("weight value")
plt.title("Ridge shrinks all weights; Lasso zeroes some out")
plt.xticks(column_numbers, feature_names, rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## Step 7 — turn the knob slowly for Ridge

Now let us watch the `alpha` knob in action. We try many values, from tiny to
large, and record the weights each time. For Ridge, every weight should slide
smoothly toward zero as `alpha` grows, but none should snap to zero.

In [ ]:
# A range of alpha values, from very small to large.
# np.logspace(-2, 3, 30) gives 30 values spread evenly on a log scale,
# from 10^-2 = 0.01 up to 10^3 = 1000.
alpha_values = np.logspace(-2, 3, 30)

# We will store the 7 weights we get for each alpha.
ridge_path = []

for one_alpha in alpha_values:
    model = Ridge(alpha=one_alpha)
    model.fit(features, target)
    ridge_path.append(model.coef_)

# Turn the list into an array so each column is one feature's path.
ridge_path = np.array(ridge_path)

print("ridge_path shape:", ridge_path.shape, "(one row per alpha, one column per feature)")

## Step 8 — plot the Ridge path

Each line is one feature's weight as `alpha` increases. The bottom axis is on a log
scale because `alpha` spans a wide range. You should see smooth curves, all heading
toward zero but never quite arriving.

In [ ]:
plt.figure(figsize=(9, 5))

# Draw one line per attribute.
for column_number in range(number_of_features):
    plt.plot(alpha_values, ridge_path[:, column_number],
             label=feature_names[column_number])

plt.xscale("log")                 # alpha covers a wide range, so use a log axis
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("alpha (penalty strength, log scale)")
plt.ylabel("weight value")
plt.title("Ridge path: weights shrink smoothly, never reaching exactly zero")
plt.legend()
plt.show()

## Step 9 — the same knob for Lasso

Same idea, now for Lasso. Here the difference is dramatic: each weight shrinks and
then hits exactly zero at some `alpha` and stays there. You can literally watch
features being switched off one by one.

In [ ]:
# A range of alpha values for Lasso (smaller numbers work well here).
lasso_alpha_values = np.logspace(-3, 1, 30)

lasso_path = []
for one_alpha in lasso_alpha_values:
    # max_iter is just how long Lasso is allowed to search; a bigger number
    # lets it settle fully for the small alphas.
    model = Lasso(alpha=one_alpha, max_iter=10000)
    model.fit(features, target)
    lasso_path.append(model.coef_)

lasso_path = np.array(lasso_path)

plt.figure(figsize=(9, 5))
for column_number in range(number_of_features):
    plt.plot(lasso_alpha_values, lasso_path[:, column_number],
             label=feature_names[column_number])

plt.xscale("log")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("alpha (penalty strength, log scale)")
plt.ylabel("weight value")
plt.title("Lasso path: weights reach exactly zero and stay there")
plt.legend()
plt.show()

### Stretch (optional) — add the penalty by hand

Skip this if you are new to code or to matrices. If you already know some linear
algebra, here is the satisfying part. Ridge's "shrink every weight" is not magic:
it comes straight from a one-line formula. The plain model solves
`(XᵀX) w = Xᵀ y`; Ridge just adds `alpha` down the diagonal first, which is the
penalty pulling the weights toward zero:

```
w = (XᵀX + alpha · I)⁻¹ Xᵀ y
```

We build that by hand and check it matches scikit-learn's Ridge to the last
decimal. (We turn off the intercept on both sides so the two are solving exactly
the same problem.)

In [ ]:
# Our synthetic prices were built with no intercept, so we drop it on both sides.
alpha_for_check = 10.0

# Build the matrix pieces. X is our features, y is the target (price).
X = features
y = target
number_of_columns = X.shape[1]
identity = np.eye(number_of_columns)      # the "I" in the formula (1s on the diagonal)

# The Ridge formula, done by hand.
weights_by_hand = np.linalg.solve(X.T @ X + alpha_for_check * identity, X.T @ y)

# scikit-learn's Ridge, with the intercept turned off so it matches.
ridge_no_intercept = Ridge(alpha=alpha_for_check, fit_intercept=False)
ridge_no_intercept.fit(X, y)

print("by hand :", np.round(weights_by_hand, 4))
print("sklearn :", np.round(ridge_no_intercept.coef_, 4))
print("\nThey match - Ridge was solving this same penalised equation all along.")

## What you just did

You saw the two big ideas of regularization in action:

- **Ridge = shrink.** Every weight is pulled toward zero smoothly, but stays in the
  model.
- **Lasso = drop.** Some weights are driven to exactly zero, dropping those
  features and giving you a simpler model.
- **ElasticNet** blends the two.

The `alpha` knob controls how firmly we turn the volume down. So far we picked
`alpha` by hand. In the next notebook we let cross-validation choose it for us on
real housing data, and we build the whole thing into one tidy project.

Next notebook: `02_housing_price_project.ipynb`.